# 01 — Minimal end-to-end loop (TF-IDF + LR)

In [1]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # so `import src...` works from notebooks/

import numpy as np
import pandas as pd

from src import config
from src.utils import set_seed, save_fig
set_seed()  # fix all RNGs -- reproducibility

Goal: light up the whole chain — load → split → **one** feature → **one** classifier → Macro-F1 + confusion. Everything else is filling this in.

In [2]:
from src import data, features, modeling

clean = data.load_or_build_clean()
splits = data.load_or_build_splits(clean)
y = clean[config.LABEL_COL].values

In [3]:
# TF-IDF only
X_tfidf, vec = features.build_tfidf(clean.iloc[splits['train']]['text'], clean['text'])
Xtr, ytr = X_tfidf[splits['train']], y[splits['train']]
Xval, yval = X_tfidf[splits['val']], y[splits['val']]

In [4]:
res = modeling.train_and_evaluate('logreg', Xtr, ytr, Xval, yval)
print('Macro-F1:', res.macro_f1)
res.per_class

Macro-F1: 0.7453099904456035


{'chatgpt': {'precision': 0.8844953173777316,
  'recall': 0.8923884514435696,
  'f1': 0.8884243532793311},
 'cohere': {'precision': 0.6429691583899634,
  'recall': 0.6470278800631246,
  'f1': 0.6449921342422653},
 'cohere-chat': {'precision': 0.6579783852511125,
  'recall': 0.5441640378548895,
  'f1': 0.5956834532374101},
 'gpt2': {'precision': 0.7121928166351607,
  'recall': 0.7994694960212202,
  'f1': 0.7533116720819795},
 'gpt3': {'precision': 0.7223624432104997,
  'recall': 0.7543489720611491,
  'f1': 0.738009283135637},
 'gpt4': {'precision': 0.9217527386541471,
  'recall': 0.9246467817896389,
  'f1': 0.9231974921630094},
 'human': {'precision': 0.7562674094707521,
  'recall': 0.8587243015287296,
  'f1': 0.8042458652184645},
 'llama-chat': {'precision': 0.8524836929252383,
  'recall': 0.8975171685155837,
  'f1': 0.8744209984559959},
 'mistral': {'precision': 0.6537313432835821,
  'recall': 0.5784469096671949,
  'f1': 0.6137892376681614},
 'mistral-chat': {'precision': 0.7536842105

Once this prints a sane Macro-F1 and confusion matrix, the skeleton works. Move on to fleshing out `train_and_evaluate` metrics, then features.